# 📏 Measuring Structural Truth: The RPF Score
### Validating Protein Structures against NMR Restraints

**Run this notebook in the cloud:** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elkins/synth-pdb/blob/master/docs/tutorials/nmr_validation_rpf.ipynb)

---

## 🎯 Objectives

In this tutorial, you will learn how to:
1.  **Define "Structural Truth"** in the context of solution NMR.
2.  **Calculate RPF Scores** (Recall, Precision, and F-measure) for a structural model.
3.  **Identify Over-fitting** by observing drops in the Precision score.
4.  **Benchmark Refinement** using real-world validation metrics from the Montelione group.


## 🧬 Background: What are RPF Scores?

NMR structure determination relies on **distance restraints** derived from the Nuclear Overhauser Effect (NOE). However, a structural model can look "perfect" geometrically (no clashes, good angles) but completely fail to represent the experimental data.

To bridge this gap, the **Montelione group** developed RPF scores (*Huang et al. 2005, JACS*):

- **Recall (R)**: Of the restraints we observed, how many are satisfied by the model? (Sensitivity)
- **Precision (P)**: Of the short distances in our model, how many are supported by our data? (Specificity)
- **F-measure (F)**: The harmonic mean of Recall and Precision. This is our single "Global Quality Score."

In [ ]:
# @title 🛠️ Setup & Installation { display-mode: "form" }
import os
import sys

# ── Environment detection ──────────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("🌐 Running in Google Colab")
    try:
        import synth_pdb  # already installed (e.g. via pip in a previous run)
        print("   ✅ synth-pdb already installed")
    except ImportError:
        print("   📦 Installing synth-pdb and dependencies...")
        # biotite is bundled as a dependency of synth-pdb; listed explicitly
        # so Colab users see exactly what is being installed.
        import subprocess
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "synth-pdb", "biotite", "matplotlib"],
            check=True,
        )
        print("   ✅ Installation complete")
else:
    # Local / CI: add the repository root so the development copy is used
    print("💻 Running in local Jupyter environment")
    _repo_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
    if _repo_root not in sys.path:
        sys.path.insert(0, _repo_root)
        print(f"   📌 Added repo root to path: {_repo_root}")

print("✅ Environment ready!")


In [ ]:
import numpy as np
import biotite.structure as struc
import biotite.structure.io.pdb as pdb
import matplotlib.pyplot as plt

from synth_pdb.generator import generate_pdb_content
from synth_pdb.nmr import calculate_rpf_score, calculate_synthetic_noes
from synth_pdb.validator import PDBValidator

print("✅ NMR Validation module loaded!")

## 🟢 Step 1: Generate a "Ground Truth" Structure

We will start by generating a high-quality, energy-minimized structure of a small protein domain and "observing" its NOEs.

In [ ]:
import io

# Generate a high-quality helix and observe its NOEs as "ground truth"
seq = "AKAAKAKAAK" * 2
pdb_content = generate_pdb_content(
    sequence_str=seq,
    minimize_energy=True,  # Physics-based refinement for realistic coordinates
)

# Parse the PDB string into a Biotite AtomArray
structure = pdb.PDBFile.read(io.StringIO(pdb_content)).get_structure(model=1)

# Derive "experimental" NOE restraints from the ground-truth geometry
restraints = calculate_synthetic_noes(structure)
print(f"Generated {len(restraints)} synthetic NOE restraints from ground truth.")
print(f"Structure: {len(structure)} atoms, sequence length {len(seq)} residues")


## 🏅 Step 2: Validating the Perfect Fit

If we test the Ground Truth structure against its own NOEs, all scores should be **1.0**.

In [ ]:
scores = calculate_rpf_score(structure, restraints)
print("--- Ground Truth Scores ---")
for k, v in scores.items():
    print(f"{k.capitalize()}: {v:.4f}")

## ⚠️ Step 3: Identifying the "Over-folded" Decoy

What happens if we "compact" the protein artificially? The Recall will remain high (as the original contacts are still there), but the **Precision** will plummet because we have created many "new" short distances that are NOT supported by the data.

In [ ]:
# Create a 'collapsed' version by scaling coordinates towards the center
coords = structure.coord
center = np.mean(coords, axis=0)
collapsed_coords = (coords - center) * 0.7 + center # 30% collapse

collapsed_structure = structure.copy()
collapsed_structure.coord = collapsed_coords

collapsed_scores = calculate_rpf_score(collapsed_structure, restraints)
print("--- Collapsed Model Scores ---")
for k, v in collapsed_scores.items():
    print(f"{k.capitalize()}: {v:.4f}")

## 📊 Step 4: Visualising the Score Collapse

Numbers are convincing — but a chart makes the story unmissable.
The bar chart below shows Recall, Precision, and F-measure for both
the Ground Truth and the over-compacted Collapsed Model.

> **Watch the Precision bar.** It drops because the collapsed structure has
> many atom pairs that are *newly* close together but unsupported by any NOE.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

metrics = ["Recall", "Precision", "F-measure"]
keys    = ["recall", "precision", "f_measure"]

ground_truth_vals = [scores.get(k, 0.0)          for k in keys]
collapsed_vals    = [collapsed_scores.get(k, 0.0) for k in keys]

x = range(len(metrics))
width = 0.35

bars_gt = ax.bar([i - width/2 for i in x], ground_truth_vals,
                 width, label="Ground Truth", color="#2ecc71", alpha=0.85)
bars_co = ax.bar([i + width/2 for i in x], collapsed_vals,
                 width, label="Collapsed Model", color="#e74c3c", alpha=0.85)

# Value labels on each bar
for bar in bars_gt:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
for bar in bars_co:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(list(x))
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score", fontsize=11)
ax.set_title("RPF Scores: Ground Truth vs Over-Compacted Model", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

plt.tight_layout()
plt.show()

print(f"\nPrecision drop: {scores.get('precision',0):.3f} → {collapsed_scores.get('precision',0):.3f} "
      f"({(scores.get('precision',0)-collapsed_scores.get('precision',0))*100:.1f} percentage points)")


## 🚀 Summary

The RPF score is a powerful way to detect **over-fitting**. In real-world structure determination, we aim to maximize the F-measure. 

- **High Recall, Low Precision**? Your protein is likely too compact or "misfolded" locally.
- **Low Recall, High Precision**? You haven't satisfied your experimental constraints yet.